# Execution Models\n\nThis notebook demonstrates a simple execution-cost framework to compare gross vs net strategy returns under basic transaction cost assumptions.\n\n## Goals\n- Create a toy signal and turnover profile\n- Apply execution cost assumptions\n- Compare gross and net performance\n- Illustrate why execution matters in realized outcomes

In [ ]:
import numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nnp.random.seed(7)

In [ ]:
# Synthetic market returns + synthetic signal\nn = 400\ndates = pd.date_range('2023-01-01', periods=n, freq='B')\n\nmarket_ret = np.random.normal(0.0002, 0.009, n)\nraw_signal = np.random.choice([-1, 0, 1], size=n, p=[0.3, 0.4, 0.3])\n\ndf = pd.DataFrame({\n    'market_ret': market_ret,\n    'signal_raw': raw_signal\n}, index=dates)\n\n# Shift signal for implementation realism\ndf['position'] = df['signal_raw'].shift(1).fillna(0)\n\n# Gross pnl\ndf['gross_ret'] = df['position'] * df['market_ret']\ndf.head()

In [ ]:
# Turnover proxy: absolute change in position\ndf['turnover'] = (df['position'] - df['position'].shift(1).fillna(0)).abs()\n\n# Cost assumptions (illustrative)\n# Example: 10 bps per unit turnover\ncost_bps_per_turn = 10\ndf['cost'] = (cost_bps_per_turn / 10000.0) * df['turnover']\n\n# Net returns after costs\ndf['net_ret'] = df['gross_ret'] - df['cost']\ndf[['position', 'turnover', 'gross_ret', 'cost', 'net_ret']].head(10)

In [ ]:
# Equity curves\ndf['gross_equity'] = (1 + df['gross_ret']).cumprod()\ndf['net_equity'] = (1 + df['net_ret']).cumprod()\n\nplt.figure(figsize=(12, 5))\nplt.plot(df.index, df['gross_equity'], label='Gross Equity')\nplt.plot(df.index, df['net_equity'], label='Net Equity (After Costs)')\nplt.title('Execution Costs: Gross vs Net Performance (Illustrative)')\nplt.xlabel('Date')\nplt.ylabel('Growth of $1')\nplt.legend()\nplt.tight_layout()\nplt.show()

In [ ]:
# Cost sensitivity analysis\ncost_grid_bps = [0, 5, 10, 15, 20, 25]\nresults = []\n\nfor bps in cost_grid_bps:\n    cost_series = (bps / 10000.0) * df['turnover']\n    net_series = df['gross_ret'] - cost_series\n    terminal = float((1 + net_series).cumprod().iloc[-1])\n    results.append({'cost_bps': bps, 'terminal_growth_of_1': terminal})\n\ncost_sensitivity = pd.DataFrame(results)\ncost_sensitivity

In [ ]:
plt.figure(figsize=(10, 4))\nplt.plot(cost_sensitivity['cost_bps'], cost_sensitivity['terminal_growth_of_1'], marker='o')\nplt.title('Terminal Performance Sensitivity to Execution Costs')\nplt.xlabel('Cost Assumption (bps per turnover unit)')\nplt.ylabel('Terminal Growth of $1')\nplt.tight_layout()\nplt.show()

## Notes\n\n- This is an **illustrative execution modeling notebook** using synthetic returns and a toy signal.\n- The purpose is to demonstrate execution-aware research thinking, not live trading claims.\n- In a production setting, execution models would include:\n  - spread-aware costs\n  - volatility scaling\n  - participation limits\n  - latency/queue effects\n  - venue-specific assumptions